In [6]:
from typing import Dict, List, Tuple

from pokerkit import HandHistory
from utils import create_state, translate_action_into_state, filter_sm_actions
from hand_to_text import INSTRUCTION_TUNED_PROMPT, get_current_situation, get_hand_background

def translate_action_to_english(action: str) -> str:
    """Translate an action into English"""
    if "cc" in action:
        return "cc"
    elif "f" in action:
        return "f"
    elif "cbr" in action:
        return "cbr"
    else:
        raise ValueError(f"Invalid action: {action}")
    
def extract_value_from_action(action: str, stack_before: int, stack_after: int) -> int:
    """Extract the value of the raise from an action"""
    if "cbr" in action:
        return 1 - (stack_after / stack_before)
    else:
        return -100
    

def convert_hand_to_narrative_instruction_tuned(
    hand_history: HandHistory, player_number: int, special_action_word: str = "ACTION"
) -> str:
    """
    This function will return three lists of strings: instruction, response, and value
    The response will be check/call, fold or raise.
    The instruction will be all of the info necessary to make a decision.
    The value will be the value of the raise if one happens - otherwise -100.
    """
    state = create_state(
        hand_history.blinds_or_straddles[0],
        hand_history.starting_stacks,
        len(hand_history.players),
    )
    base_instruction = INSTRUCTION_TUNED_PROMPT + "\n" + get_hand_background(hand_history, player_number)
    filtered_actions = filter_sm_actions(hand_history.actions)

    instruction = []
    response = []
    value = []

    for action in filtered_actions:
        if action.startswith("d dh"):
            state = translate_action_into_state(action, state)
            continue
        if action.startswith(f"p{player_number}"):
            current_situation = get_current_situation(state, player_number)
            instruction.append(f"{base_instruction}\n{current_situation}")
            response.append(translate_action_to_english(action))
            stack_before = state.stacks[player_number - 1]
            
        base_instruction = f"{base_instruction}\n{action}"
        state = translate_action_into_state(action, state)
        if action.startswith(f"p{player_number}"):
            stack_after = state.stacks[player_number - 1]
            value.append(extract_value_from_action(action, stack_before, stack_after))

    return instruction, response, value


In [8]:
import random

import pandas as pd
from hand_to_text import convert_hand_to_narrative2
from pokerbench_translator import create_pokerkit_state_postflop
from utils import create_state, verify_hand
import tqdm

df = pd.read_csv("/Users/derek/Desktop/poker_datasets/datasets/postflop_10k_test_set_game_scenario_information.csv")


random_number = random.randint(0, len(df))
print(df.iloc[random_number]['preflop_action'])
print(df.iloc[random_number]['postflop_action'])
print(df.iloc[random_number]['correct_decision'])
print(df.iloc[random_number]['hero_position'])
print(df.iloc[random_number], "\n\n\n")


issue_hands = []
# for i in tqdm.tqdm(range(len(df))):
#     idx = i
#     hand_history, player_number = create_pokerkit_state_postflop(df.iloc[idx])
#     if not verify_hand(hand_history.actions, hand_history.blinds_or_straddles[0], hand_history.starting_stacks):
#         issue_hands.append(idx)
#     else:
#         for player_number in range(1, 7):
#             instruction, response, value = convert_hand_to_narrative_instruction_tuned(hand_history, player_number)
hand_history, player_number = create_pokerkit_state_postflop(df.iloc[random_number])

instruction, response, value = convert_hand_to_narrative_instruction_tuned(hand_history, player_number)

for i in range(len(instruction)):
    print(f"Instruction: {instruction[i]}")
    print(f"Response: {response[i]}")
    print(f"Value: {value[i]}\n\n\n")

UTG/2.0bb/BB/call
OOP_CHECK/IP_CHECK/dealcards/6h/OOP_BET_5/IP_RAISE_14/OOP_CALL/dealcards/4h/OOP_CHECK/IP_BET_24
Call
OOP
Unnamed: 0                                                         5011
preflop_action                                        UTG/2.0bb/BB/call
board_flop                                                       Ks7h2d
board_turn                                                           6h
board_river                                                          4h
aggressor_position                                                  OOP
postflop_action       OOP_CHECK/IP_CHECK/dealcards/6h/OOP_BET_5/IP_R...
evaluation_at                                                     River
available_moves                            ['Fold', 'Call', 'Raise 84']
pot_size                                                             56
hero_position                                                       OOP
holding                                                            Kh5h
correct_decis

In [20]:
import random

import pandas as pd
from hand_to_text import convert_hand_to_narrative2
from pokerbench_translator import create_pokerkit_state_postflop, create_pokerkit_state_preflop
from utils import create_state, verify_hand
import tqdm

df = pd.read_csv("/Users/derek/Desktop/poker_datasets/datasets/preflop_60k_train_set_game_scenario_information.csv")


random_number = random.randint(0, len(df))
# print(df.iloc[random_number]['preflop_action'])
# print(df.iloc[random_number]['postflop_action'])
# print(df.iloc[random_number]['correct_decision'])
# print(df.iloc[random_number]['hero_position'])
# print(df.iloc[random_number], "\n\n\n")


issue_hands = []
# for i in tqdm.tqdm(range(len(df))):
#     idx = i
#     hand_history, player_number = create_pokerkit_state_preflop(df.iloc[idx])
#     if not verify_hand(hand_history.actions, hand_history.blinds_or_straddles[0], hand_history.starting_stacks):
#         issue_hands.append(idx)
#     else:
#         for player_number in range(1, 7):
#             instruction, response, value = convert_hand_to_narrative_instruction_tuned(hand_history, player_number)
#     break
# hand_history, player_number = create_pokerkit_state_postflop(df.iloc[random_number])

idx = random_number
hand_history, player_number = create_pokerkit_state_preflop(df.iloc[idx])
instruction, response, value = convert_hand_to_narrative_instruction_tuned(hand_history, player_number)

for i in range(len(instruction)):
    print(f"Instruction: {instruction[i]}")
    print(f"Response: {response[i]}")
    print(f"Value: {value[i]}\n\n\n")

Instruction: You are an expert poker player tasked with making a decision. Your choices are to cc (check/call), f (fold), or cbr (raise). You must respond with a singe phrase: 'cc', 'f', or 'cbr'.

There are 6 players at the table, I am player 4.
The starting stacks: 100, 100, 100, 100, 100, 100.
The current small blind and big blind is 0.5 and 1.
My cards: jack of spades, ten of diamonds. And the following is the action sequence if any actions have happened:
p3 fold
Pot size: 1.5, needed bet: 1.0, minimum raise: 2.0, my stack: 100
Response: cbr
Value: 2.0



Instruction: You are an expert poker player tasked with making a decision. Your choices are to cc (check/call), f (fold), or cbr (raise). You must respond with a singe phrase: 'cc', 'f', or 'cbr'.

There are 6 players at the table, I am player 4.
The starting stacks: 100, 100, 100, 100, 100, 100.
The current small blind and big blind is 0.5 and 1.
My cards: jack of spades, ten of diamonds. And the following is the action sequence 